In [1]:
import os 
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np

In [3]:
# Load data
model_input = pd.read_parquet("/data/big/fmoss/data/model_input_dataset/training_dataset.parquet")
# Add year column based on DisNo.
model_input["year"] = model_input["DisNo."].apply(lambda x: int(x[:4]))

In [4]:
# # Filtering
# df = model_input.copy()
# # People affected > 100
# events_to_consider = df[df["Total Affected"] > 100]["DisNo."].unique()
# df = df[df["DisNo."].isin(events_to_consider)]
# # Only subnational reported events
# df = df[df.level != "ADM0"]

df = model_input.copy()

# People affected > 100
events_to_consider_c1 = df[df["Total Affected"] > 100]["DisNo."].unique()
events_to_consider_c2 = df[df["level"] != "ADM0"]["DisNo."].unique()

# More constraints regarding subnational reported events
proportions = (
    df[df["level"] != "ADM0"]
    .assign(is_affected=lambda x: x["Total Affected"] > 0)
    .groupby(["DisNo.", "GID_0", "GID_1"])["is_affected"].max()
    .groupby(["DisNo.", "GID_0"]).mean()
    .reset_index(name="proportion_affected_gid1")
)

events_to_consider_c3 = proportions[proportions.proportion_affected_gid1 < 1]["DisNo."].unique()

events_to_consider = list(set(events_to_consider_c1) & set(events_to_consider_c2) & set(events_to_consider_c3))
print(len(events_to_consider))

df = df[df["DisNo."].isin(events_to_consider)]

780


In [5]:
# missing events for the xgb model
# missing_events = [
#     '2003-0346-CHN',
#     '2003-0346-HKG',
#     '2003-0443-CHN',
#     '2006-0410-PHL',
#     '2006-0415-PHL',
#     '2007-0457-PHL',
#     '2007-0668-PHL',
#     '2008-0329-CHN',
#     '2008-0369-CHN',
#     '2008-0369-HKG',
#     '2009-0399-CHN',
#     '2009-0399-HKG',
#     '2017-0352-CHN',
#     '2017-0352-HKG'
#     ]

missing_events = [
    '2009-0399-CHN',
    '2003-0346-CHN',
    '2003-0346-HKG',
    '2017-0352-HKG',
    '2009-0399-HKG',
    '2008-0369-CHN',
    '2007-0457-PHL',
    '2006-0410-PHL',
    '2017-0352-CHN',
    '2008-0369-HKG',
    '2006-0415-PHL',
    '2007-0668-PHL'
    ]

df = df[~df["DisNo."].isin(missing_events)]

In [6]:
print(len(df["DisNo."].unique()))
print(len(df["GID_0"].unique()))

768
67


In [7]:
# # Naive model based on windspeed
# def naive_model_windspeed(df, threshold=33, level="ADM0"):
#     """
#     Compute a windspeed-based impact model at ADM0 or ADM1 level based on windspeed threshold.
#     Grid cells with windspeed above threshold get 100% impact (all population affected).

#     Parameters:
#     - model_input: pd.DataFrame with required columns including 'windspeed'
#     - level: str, either 'ADM0' or 'ADM1'
#     - windspeed_threshold: float, windspeed threshold in m/s (default 33 m/s ~= 120 km/h ~= 64 knots == category 1 hurricane on saffir-simpson scale)

#     Returns:
#     - pd.DataFrame with predicted vs actual impacts and error metrics.
#     """
#     df = df.copy()
#     # Compute predicted flag: 1 if wind_speed > threshold else 0
#     df["predicted_flag"] = (df["wind_speed"] > threshold).astype(int)

#     # Compute predicted population: population if flagged, else 0
#     df["predicted"] = df["population"] * df["predicted_flag"]

#     # Compute reported population: percentage of population affected
#     df["reported"] = df["population"] * df["perc_affected_pop_grid_region"] / 100

#     # Select final columns
#     df_predictions = df[
#         ["DisNo.", "GID_0", "GID_1", "id", "level", "population", 
#         "wind_speed", "Total Affected", "predicted", "reported"]
#     ].copy()
    
#     # Grouping and Aggregation
#     if level == "ADM0":
#         group_keys = ["DisNo.", "GID_0"]
#         grouped = df_predictions.groupby(group_keys).agg(
#             prediction_ppl=("predicted", "sum"),
#             actual_ppl=("reported", "sum"),
#             population=("population", lambda x: x[df_predictions.loc[x.index, "Total Affected"] > 0].sum()),
#             total_affected=("Total Affected", "max"),
#             wind_speed=("wind_speed", "max")
#         ).reset_index()
#         grouped["reported"] = grouped["total_affected"] / grouped["population"]
#         grouped["reported"] = grouped["reported"].clip(upper=100) # Some cases in here they report > 100% population affected. 
#                                                                   # We played with this when disagreggating and capping to 100% 
#                                                                   # Since we are using Total Affection again, we need to cap again

#     elif level == "ADM1":
#         group_keys = ["DisNo.", "GID_0", "GID_1", "level"]
#         grouped = df_predictions.groupby(group_keys).agg(
#             prediction_ppl=("predicted", "sum"),
#             actual_ppl=("reported", "sum"),
#             population=("population", "sum"),
#             total_affected=("Total Affected", "max"),
#             wind_speed=("wind_speed", "max")
#         ).reset_index()
#         grouped["reported"] = 100 * grouped["actual_ppl"] / grouped["population"]
#         grouped["reported"] = grouped["reported"].clip(upper=100)
#     else:
#         raise ValueError("Level must be 'ADM0' or 'ADM1'.")
    
#     # Compute percentages
#     grouped["predicted"] = 100 * grouped["prediction_ppl"] / grouped["population"]
#     grouped.rename(columns={"total_affected": "Total Affected"}, inplace=True)

#     # Cap predictions to a maximum of 100% (already at 100% by design, but keeping for consistency)
#     grouped.loc[grouped.predicted > 100, "predicted"] = 100

#     # Add error metrics
#     grouped["error"] = grouped["predicted"] - grouped["reported"]
#     grouped["abs_error"] = abs(grouped["error"])

#     return grouped.reset_index(drop=True)




def compute_historical_predictions(df, threshold=33):
    # Precompute for each country + year + event the median of previous % impacts
    # First build a mapping of country -> list of (year, DisNo., median)
    country_events = (
        df.groupby(["GID_0", "DisNo.", "year"])
        ["perc_affected_pop_grid_region"]
        .apply(lambda x: x.unique())
        .reset_index()
    )

    # Remove zeros in the unique percents
    country_events["impact_percent"] = country_events["perc_affected_pop_grid_region"].apply(
        lambda arr: [v for v in arr if v != 0]
    )

    # We don't need the original array anymore
    country_events.drop(columns="perc_affected_pop_grid_region", inplace=True)

    # # Build lookup table: for each (GID_0, year, DisNo.), the median of past events
    # result_rows = []
    # for _, row in country_events.iterrows():
    #     gid0 = row["GID_0"]
    #     disno = row["DisNo."]
    #     year = row["year"]

    #     # Get all past events for this country with year <= current year, and different event
    #     past = country_events[
    #         (country_events["GID_0"] == gid0)
    #         & (country_events["year"] <= year)
    #         & (country_events["DisNo."] != disno)
    #     ]

    #     # Collect all impact percentages from past
    #     past_impacts = [val for sublist in past["impact_percent"] for val in sublist]

    #     if past_impacts:
    #         median_pct = np.median(past_impacts)
    #     else:
    #         median_pct = 0

    #     result_rows.append((gid0, disno, median_pct))

    # # Build a dataframe for quick merging
    # median_df = pd.DataFrame(result_rows, columns=["GID_0", "DisNo.", "median_pct"])

    # # Merge back to original df
    # df = df.merge(median_df, on=["GID_0", "DisNo."], how="left")

    # Compute predictions
    df["predicted_flag"] = (df["wind_speed"] > threshold).astype(int)
    df["predicted"] = (100.0) * df["population"] * df["predicted_flag"]
    df["reported_ppl"] = df["population"] * df["perc_affected_pop_grid_region"] / 100
    df["reported"] = df["perc_affected_pop_grid_region"]

    # Final selection
    df_predictions = df[
        ["DisNo.", "GID_0", "GID_1", "id", "level", "population",
         "wind_speed", "Total Affected", "predicted", "reported", "reported_ppl"]
    ].copy()

    return df_predictions


# def compute_grouped_predictions(df_predictions, level="ADM0"):
#     if level == "ADM0":
#         group_keys = ["DisNo.", "GID_0"]
#         grouped = (
#             df_predictions.groupby(group_keys)
#             .agg(
#                 prediction_ppl=("predicted", "sum"),
#                 actual_ppl=("reported", "sum"),
#                 # population=("population", lambda x: x[df_predictions.loc[x.index, "Total Affected"] > 0].sum()),
#                 population=("population", "sum"),
#                 total_affected=("Total Affected", "max"),
#                 wind_speed=("wind_speed", "max"),
#             )
#             .reset_index()
#         )
#         grouped["reported"] = 100 * grouped["total_affected"] / grouped["population"]
#         grouped["reported"] = grouped["reported"].clip(upper=100) # Some cases in here they report > 100% population affected. 
#                                                                   # We played with this when disagreggating and capping to 100% 
#                                                                   # Since we are using Total Affection again, we need to cap again
#     elif level == "ADM1":
#         group_keys = ["DisNo.", "GID_0", "GID_1", "level"]
#         grouped = (
#             df_predictions.groupby(group_keys)
#             .agg(
#                 prediction_ppl=("predicted", "sum"),
#                 actual_ppl=("reported", "sum"),
#                 population=("population", "sum"),
#                 total_affected=("Total Affected", "max"),
#                 wind_speed=("wind_speed", "max"),
#             )
#             .reset_index()
#         )
#         grouped["reported"] = 100 * grouped["actual_ppl"] / grouped["population"]
#         grouped["reported"] = grouped["reported"].clip(upper=100)
#     else:
#         raise ValueError("Level must be 'ADM0' or 'ADM1'.")

#     # Compute predicted percentage
#     grouped["predicted"] = 100 * grouped["prediction_ppl"] / grouped["population"]

#     # Cap predictions at 100
#     grouped["predicted"] = grouped["predicted"].clip(upper=100)

#     # Compute errors
#     grouped["error"] = grouped["predicted"] - grouped["reported"]
#     grouped["abs_error"] = grouped["error"].abs()

#     # Rename total_affected for clarity
#     grouped.rename(columns={"total_affected": "Total Affected"}, inplace=True)

#     return grouped.reset_index(drop=True)



def compute_grouped_predictions(df_test):
    group_keys = ["DisNo.", "GID_0", "GID_1"]
    grouped = (
        df_test.groupby(group_keys)
        .agg(
            prediction_ppl=("predicted", "sum"),
            actual_ppl=("reported_ppl", "sum"),
            actual_perc=("reported", "max"),
            population=("population", "sum")
        )
        .reset_index()
    )
    grouped["reported"] = grouped["actual_perc"] 
    grouped["reported"] = grouped["reported"].clip(upper=100)


    # Compute predicted percentage
    grouped["predicted"] = 100 * grouped["prediction_ppl"] / grouped["population"]

    # Cap predictions at 100
    grouped["predicted"] = grouped["predicted"].clip(upper=100)

    # Compute errors
    grouped["error"] = grouped["predicted"] - grouped["reported"]
    grouped["abs_error"] = grouped["error"].abs()

    return grouped.reset_index(drop=True)

    

In [8]:
wind_based_model = compute_historical_predictions(df)

In [9]:
# Windspeed exposed model
# wind_model_adm0 = compute_grouped_predictions(wind_based_model, "ADM0")
# wind_model_adm1 = compute_grouped_predictions(wind_based_model, "ADM1")
wind_model_adm1 = compute_grouped_predictions(wind_based_model)

In [12]:
wind_model_adm1[wind_model_adm1.reported >= 15].shape

(613, 11)

In [9]:
# Save model outputs
# wind_model_adm0.to_parquet("/data/big/fmoss/data/model_output/merged/adm0_grouped/wind_naive_model.parquet", index=False)
wind_model_adm1.to_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/wind_naive_model.parquet", index=False)

In [10]:
# Naive model based on just historical data
def compute_naive_historical_predictions(df):
    # Precompute for each country + year + event the median of previous % impacts
    # First build a mapping of country -> list of (year, DisNo., impact %)
    country_events = (
        df.groupby(["GID_0", "DisNo.", "year"])
        ["perc_affected_pop_grid_region"]
        .apply(lambda x: x.unique())
        .reset_index()
    )

    # Remove zeros in the unique percents
    country_events["impact_percent"] = country_events["perc_affected_pop_grid_region"].apply(
        lambda arr: [v for v in arr if v != 0]
    )

    # Drop original array
    country_events.drop(columns="perc_affected_pop_grid_region", inplace=True)

    # Build lookup table: (GID_0, DisNo.) -> median of past events' % impact
    result_rows = []
    for _, row in country_events.iterrows():
        gid0 = row["GID_0"]
        disno = row["DisNo."]
        year = row["year"]

        # All past events (same country, year <= current, different event)
        past = country_events[
            (country_events["GID_0"] == gid0)
            & (country_events["year"] <= year)
            & (country_events["DisNo."] != disno)
        ]

        # Gather past impact percentages
        past_impacts = [val for sublist in past["impact_percent"] for val in sublist]

        if past_impacts:
            median_pct = np.median(past_impacts)
        else:
            median_pct = 0

        result_rows.append((gid0, disno, median_pct))

    median_df = pd.DataFrame(result_rows, columns=["GID_0", "DisNo.", "median_pct"])

    # Merge median % back into df
    df = df.merge(median_df, on=["GID_0", "DisNo."], how="left")

    # Compute prediction: assign median % to *all* grid cells
    df["predicted"] = (df["median_pct"] / 100.0) * df["population"]
    df["reported_ppl"] = (df["perc_affected_pop_grid_region"] / 100.0) * df["population"]
    df["reported"] = df["perc_affected_pop_grid_region"]

    # Final selection
    df_predictions = df[
        ["DisNo.", "GID_0", "GID_1", "id", "level", "population",
         "wind_speed", "Total Affected", "predicted", "reported", "reported_ppl"]
    ].copy()

    return df_predictions

# def compute_naive_grouped_predictions(df_predictions, level="ADM0"):
#     if level == "ADM0":
#         group_keys = ["DisNo.", "GID_0"]
#         grouped = (
#             df_predictions.groupby(group_keys)
#             .agg(
#                 prediction_ppl=("predicted", "sum"),
#                 actual_ppl=("reported", "sum"),
#                 # population=("population", lambda x: x[df_predictions.loc[x.index, "Total Affected"] > 0].sum()),
#                 population=("population", "sum"),
#                 total_affected=("Total Affected", "max"),
#                 wind_speed=("wind_speed", "max"),
#             )
#             .reset_index()
#         )
#         grouped["reported"] = 100 * grouped["total_affected"] / grouped["population"]
#         grouped["reported"] = grouped["reported"].clip(upper=100) # Some cases in here they report > 100% population affected. 
#                                                                   # We played with this when disagreggating and capping to 100% 
#                                                                   # Since we are using Total Affection again, we need to cap again
#     elif level == "ADM1":
#         group_keys = ["DisNo.", "GID_0", "GID_1", "level"]
#         grouped = (
#             df_predictions.groupby(group_keys)
#             .agg(
#                 prediction_ppl=("predicted", "sum"),
#                 actual_ppl=("reported", "sum"),
#                 population=("population", "sum"),
#                 total_affected=("Total Affected", "max"),
#                 wind_speed=("wind_speed", "max"),
#             )
#             .reset_index()
#         )
#         grouped["reported"] = 100 * grouped["actual_ppl"] / grouped["population"]
#         grouped["reported"] = grouped["reported"].clip(upper=100)
#     else:
#         raise ValueError("Level must be 'ADM0' or 'ADM1'.")

#     # Compute predicted percentage
#     grouped["predicted"] = 100 * grouped["prediction_ppl"] / grouped["population"]

#     # Cap predictions at 100
#     grouped["predicted"] = grouped["predicted"].clip(upper=100)

#     # Compute errors
#     grouped["error"] = grouped["predicted"] - grouped["reported"]
#     grouped["abs_error"] = grouped["error"].abs()

#     # Rename total_affected for clarity
#     grouped.rename(columns={"total_affected": "Total Affected"}, inplace=True)

#     return grouped.reset_index(drop=True)


In [ ]:
# # Naive windspeed-based model
# naive_model_adm0 = naive_model_windspeed(
#     df=df,
#     threshold=33,
#     level="ADM0"
# )

# naive_model_adm1 = naive_model_windspeed(
#     df=df,
#     threshold=33,
#     level="ADM1"
# )

In [11]:
# Naive historical based model
naive_model = compute_naive_historical_predictions(df)

In [14]:
# naive_model_adm0_historical = compute_naive_grouped_predictions(naive_model, "ADM0")
# naive_model_adm1_historical = compute_naive_grouped_predictions(naive_model, "ADM1")
naive_model_adm1_historical = compute_grouped_predictions(naive_model)

In [17]:
# # Save model outputs
# naive_model_adm0.to_parquet("/data/big/fmoss/data/model_output/merged/adm0_grouped/wind_naive_model.parquet", index=False)
# naive_model_adm1.to_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/wind_naive_model.parquet", index=False)

# naive_model_adm0_historical.to_parquet("/data/big/fmoss/data/model_output/merged/adm0_grouped/naive_historical_model.parquet", index=False)
naive_model_adm1_historical.to_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/naive_historical_model.parquet", index=False)

In [2]:
# Load
naive_model_adm0 = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm0_grouped/wind_naive_model.parquet")
naive_model_adm1 = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/wind_naive_model.parquet")

naive_model_adm0_historical = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm0_grouped/naive_historical_model.parquet")
naive_model_adm1_historical = pd.read_parquet("/data/big/fmoss/data/model_output/merged/adm1_grouped/naive_historical_model.parquet")

Metrics

In [3]:
from sklearn.metrics import cohen_kappa_score, f1_score, precision_score, recall_score, confusion_matrix
import numpy as np
import pandas as pd

def safe_quadratic_weighted_kappa(y_true, y_pred):
    unique_true = np.unique(y_true)
    unique_pred = np.unique(y_pred)
    
    if len(unique_true) < 2 or len(unique_pred) < 2:
        # Not enough categories to compute QWK
        return 1
    
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

def evaluate_predictions(df):
    df = df.copy()
    # Compute error metrics
    mae = df["abs_error"].mean()
    std_error = df["abs_error"].std()

    # Define category bins & labels
    bins = [-np.inf, 1, 5, 15, np.inf]
    labels = [0, 1, 2, 3]  # 0: very low, 1: low, 2: moderate, 3: high

    # Categorize reported and predicted
    df["reported_cat"] = pd.cut(df["reported"], bins=bins, labels=labels).astype(int)
    df["predicted_cat"] = pd.cut(df["predicted"], bins=bins, labels=labels).astype(int)

    # Quadratic weighted kappa
    kappa = cohen_kappa_score(df["reported_cat"], df["predicted_cat"], weights="quadratic")
    # kappa = safe_quadratic_weighted_kappa(df["reported_cat"], df["predicted_cat"])
    accuracy = (df["reported_cat"] == df["predicted_cat"]).mean()
    # Binarize: 0,1 → 0 (low); 2,3 → 1 (moderate/high)
    df["reported_bin"] = (df["reported_cat"] >= 2).astype(int)
    df["predicted_bin"] = (df["predicted_cat"] >= 2).astype(int)

    # F1, precision, recall
    f1 = f1_score(df["reported_bin"], df["predicted_bin"], zero_division=np.nan)
    precision = precision_score(df["reported_bin"], df["predicted_bin"], zero_division=np.nan)
    recall = recall_score(df["reported_bin"], df["predicted_bin"], zero_division=np.nan)

    cm = confusion_matrix(df["reported_bin"], df["predicted_bin"], labels=[0,1])
    tn, fp, fn, tp = cm.ravel()

    # Negative precision (NPV)
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    # Negative recall (specificity / TNR)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    # Helper to round to 2 significant digits
    def sig(x):
        return float(f"{x:.3g}")

    # Return metrics rounded to 2 significant digits
    return {
        "mae": sig(mae),
        "std_error": sig(std_error),
        "quadratic_weighted_kappa": sig(kappa),
        "accuracy": sig(accuracy),
        "precision": sig(precision),
        "recall": sig(recall),
        "f1": sig(f1),
        "NPV": sig(npv),
        "specificity": sig(specificity)
    }


Naive windspeed based model

In [4]:
metrics_adm0 = evaluate_predictions(naive_model_adm0)
print(metrics_adm0)

{'mae': 13.6, 'std_error': 29.4, 'quadratic_weighted_kappa': 0.0951, 'accuracy': 0.658, 'precision': 0.049, 'recall': 0.588, 'f1': 0.0905, 'NPV': 0.988, 'specificity': 0.75}


In [5]:
metrics_adm1 = evaluate_predictions(naive_model_adm1.dropna()) # there are 4 adm1 regions with no population
print(metrics_adm1)

{'mae': 2.42, 'std_error': 12.8, 'quadratic_weighted_kappa': 0.336, 'accuracy': 0.919, 'precision': 0.303, 'recall': 0.332, 'f1': 0.317, 'NPV': 0.978, 'specificity': 0.975}


Naive historical based model

In [6]:
metrics_adm0_historical = evaluate_predictions(naive_model_adm0_historical)
print(metrics_adm0_historical)

{'mae': 25.5, 'std_error': 33.4, 'quadratic_weighted_kappa': 0.222, 'accuracy': 0.332, 'precision': 0.374, 'recall': 0.72, 'f1': 0.493, 'NPV': 0.813, 'specificity': 0.503}


In [7]:
metrics_adm1_historical = evaluate_predictions(naive_model_adm1_historical.dropna())
print(metrics_adm1_historical)

{'mae': 4.2, 'std_error': 11.5, 'quadratic_weighted_kappa': 0.0359, 'accuracy': 0.431, 'precision': 0.0506, 'recall': 0.2, 'f1': 0.0808, 'NPV': 0.971, 'specificity': 0.878}


## Post-analysis: top xx events

Performance on 100 most impacting events?

In [8]:
top_impacting_events = naive_model_adm0_historical.sort_values("reported")[-100:]["DisNo."].unique()

In [9]:
top_events_adm0 = naive_model_adm0_historical[naive_model_adm0_historical["DisNo."].isin(top_impacting_events)]
top_events_adm1 = naive_model_adm1_historical[naive_model_adm1_historical["DisNo."].isin(top_impacting_events)]

metrics_adm0 = evaluate_predictions(top_events_adm0)
print(metrics_adm0)

metrics_adm1 = evaluate_predictions(top_events_adm1)
print(metrics_adm1)


{'mae': 38.6, 'std_error': 33.8, 'quadratic_weighted_kappa': 0.0, 'accuracy': 0.66, 'precision': 1.0, 'recall': 0.76, 'f1': 0.864, 'NPV': 0.0, 'specificity': nan}
{'mae': 11.0, 'std_error': 24.6, 'quadratic_weighted_kappa': -0.0176, 'accuracy': 0.224, 'precision': 0.078, 'recall': 0.133, 'f1': 0.0982, 'NPV': 0.931, 'specificity': 0.882}


In [10]:
top_events_adm0 = naive_model_adm0[naive_model_adm0["DisNo."].isin(top_impacting_events)]
top_events_adm1 = naive_model_adm1[naive_model_adm1["DisNo."].isin(top_impacting_events)]

metrics_adm0 = evaluate_predictions(top_events_adm0)
print(metrics_adm0)

metrics_adm1 = evaluate_predictions(top_events_adm1)
print(metrics_adm1)

{'mae': 49.4, 'std_error': 43.5, 'quadratic_weighted_kappa': 0.00692, 'accuracy': 0.24, 'precision': 0.164, 'recall': 0.588, 'f1': 0.256, 'NPV': 0.821, 'specificity': 0.386}
{'mae': 7.59, 'std_error': 23.0, 'quadratic_weighted_kappa': 0.348, 'accuracy': 0.857, 'precision': 0.311, 'recall': 0.424, 'f1': 0.359, 'NPV': 0.955, 'specificity': 0.929}


Performance on 100 less impacting events

In [7]:
top_less_impacting_events = naive_model_adm0_historical.sort_values("reported")[:100]["DisNo."].unique()

In [8]:
top_events_adm0 = naive_model_adm0_historical[naive_model_adm0_historical["DisNo."].isin(top_less_impacting_events)]
top_events_adm1 = naive_model_adm1_historical[naive_model_adm1_historical["DisNo."].isin(top_less_impacting_events)]

metrics_adm0 = evaluate_predictions(top_events_adm0)
print(metrics_adm0)

metrics_adm1 = evaluate_predictions(top_events_adm1)
print(metrics_adm1)

print(len(top_events_adm0[(top_events_adm0.reported <= 5) & (top_events_adm0.predicted < 5)]), "out of 100") # Perfect binary modelmodel
print(len(top_events_adm0[(top_events_adm0.reported <= 5) & (top_events_adm0.predicted > 5)]), "out of 100") # Worst possible binary modelmodel

{'mae': 9.67, 'std_error': 20.8, 'quadratic_weighted_kappa': 0.0, 'accuracy': 0.44, 'precision': 0.0, 'recall': nan, 'f1': 0.0, 'NPV': 1.0, 'specificity': 0.69}
{'mae': 2.37, 'std_error': 9.07, 'quadratic_weighted_kappa': 0.0, 'accuracy': 0.638, 'precision': 0.0, 'recall': nan, 'f1': 0.0, 'NPV': 1.0, 'specificity': 0.895}
69 out of 100
31 out of 100


In [13]:
print(len(top_events_adm0))
print(len(top_events_adm0.GID_0.unique()))

100
21


In [9]:
top_events_adm0 = naive_model_adm0[naive_model_adm0["DisNo."].isin(top_less_impacting_events)]
top_events_adm1 = naive_model_adm1[naive_model_adm1["DisNo."].isin(top_less_impacting_events)]

metrics_adm0 = evaluate_predictions(top_events_adm0)
print(metrics_adm0)

metrics_adm1 = evaluate_predictions(top_events_adm1)
print(metrics_adm1)

print(len(top_events_adm0[(top_events_adm0.reported < 5) & (top_events_adm0.predicted < 5)]), "out of 100") # Perfect binaryn modelmodel

{'mae': 6.44, 'std_error': 20.2, 'quadratic_weighted_kappa': 0.0, 'accuracy': 0.82, 'precision': 0.0, 'recall': nan, 'f1': 0.0, 'NPV': 1.0, 'specificity': 0.87}
{'mae': 0.64, 'std_error': 7.11, 'quadratic_weighted_kappa': 0.0, 'accuracy': 0.984, 'precision': 0.0, 'recall': nan, 'f1': 0.0, 'NPV': 1.0, 'specificity': 0.988}
87 out of 100


Performance on top 100 events with highest recorded windspeed

In [15]:
top_windspeed_events = naive_model_adm0_historical.sort_values("wind_speed")[-100:]["DisNo."].unique()

In [16]:
len(set(top_impacting_events) & set(top_windspeed_events))

30

In [28]:
top_windspeed_events_adm0 = naive_model_adm0_historical[naive_model_adm0_historical["DisNo."].isin(top_windspeed_events)]
top_windspeed_events_adm1 = naive_model_adm1_historical[naive_model_adm1_historical["DisNo."].isin(top_windspeed_events)]

metrics_adm0 = evaluate_predictions(top_windspeed_events_adm0)
print(metrics_adm0)

metrics_adm1 = evaluate_predictions(top_windspeed_events_adm1.dropna())
print(metrics_adm1)

{'mae': 25.0, 'std_error': 41.0, 'quadratic_weighted_kappa': 0.35, 'precision': 0.68, 'recall': 0.72, 'f1': 0.7}
{'mae': 6.8, 'std_error': 19.0, 'quadratic_weighted_kappa': 0.072, 'precision': 0.17, 'recall': 0.21, 'f1': 0.19}


In [30]:
top_windspeed_events_adm0 = naive_model_adm0[naive_model_adm0["DisNo."].isin(top_windspeed_events)]
top_windspeed_events_adm1 = naive_model_adm1[naive_model_adm1["DisNo."].isin(top_windspeed_events)]

metrics_adm0 = evaluate_predictions(top_windspeed_events_adm0)
print(metrics_adm0)

metrics_adm1 = evaluate_predictions(top_windspeed_events_adm1.dropna())
print(metrics_adm1)

{'mae': 54.0, 'std_error': 41.0, 'quadratic_weighted_kappa': 0.041, 'precision': 0.073, 'recall': 1.0, 'f1': 0.14}
{'mae': 11.0, 'std_error': 28.0, 'quadratic_weighted_kappa': 0.46, 'precision': 0.39, 'recall': 0.56, 'f1': 0.46}


In [27]:
top_windspeed_events_adm0.sort_values("wind_speed").iloc[0].wind_speed

49.86637092222806

Rich countries vs poor countries

In [15]:
gdp = pd.read_excel("/data/big/fmoss/data/GDP/CLASS.xlsx")

emdat_countries = df.iso3.unique()
print(len(emdat_countries))

gdp = gdp[["Code", "Income group"]].rename({"Code": "GID_0"}, axis=1)
gdp = gdp[gdp.GID_0.isin(emdat_countries)]
print(len(gdp))

68
67


In [16]:
naive_model_adm0 = naive_model_adm0.merge(gdp, how="left")
naive_model_adm0["Income group"] = naive_model_adm0["Income group"].fillna("Upper middle income")

naive_model_adm1 = naive_model_adm1.merge(gdp, how="left")
naive_model_adm1["Income group"] = naive_model_adm1["Income group"].fillna("Upper middle income")

naive_model_adm0_historical = naive_model_adm0_historical.merge(gdp, how="left")
naive_model_adm0_historical["Income group"] = naive_model_adm0_historical["Income group"].fillna("Upper middle income")

naive_model_adm1_historical = naive_model_adm1_historical.merge(gdp, how="left")
naive_model_adm1_historical["Income group"] = naive_model_adm1_historical["Income group"].fillna("Upper middle income")

rich_countries_events = naive_model_adm0[
    naive_model_adm0["Income group"].isin(["High income", "Upper middle income"])
    ]["DisNo."].unique()

poor_countries_events = naive_model_adm0[
    ~naive_model_adm0["Income group"].isin(["High income", "Upper middle income"])
    ]["DisNo."].unique()


print(len(rich_countries_events))
print(len(poor_countries_events))

408
385


In [17]:
# RICH subset --Naive Wind model
top_events_adm0 = naive_model_adm0[naive_model_adm0["DisNo."].isin(rich_countries_events)]
top_events_adm1 = naive_model_adm1[naive_model_adm1["DisNo."].isin(rich_countries_events)]

metrics_adm0 = evaluate_predictions(top_events_adm0)
print(metrics_adm0)

metrics_adm1 = evaluate_predictions(top_events_adm1.dropna())
print(metrics_adm1)

print(len(top_events_adm0.GID_0.unique()))
print(len(rich_countries_events))

{'mae': 15.4, 'std_error': 30.6, 'quadratic_weighted_kappa': 0.0866, 'accuracy': 0.613, 'precision': 0.0513, 'recall': 0.667, 'f1': 0.0952, 'NPV': 0.99, 'specificity': 0.722}
{'mae': 3.03, 'std_error': 15.0, 'quadratic_weighted_kappa': 0.336, 'accuracy': 0.929, 'precision': 0.266, 'recall': 0.461, 'f1': 0.338, 'NPV': 0.986, 'specificity': 0.967}
40
408


In [18]:
# RICH subset --Naive historical model
top_events_adm0 = naive_model_adm0_historical[naive_model_adm0_historical["DisNo."].isin(rich_countries_events)]
top_events_adm1 = naive_model_adm1_historical[naive_model_adm1_historical["DisNo."].isin(rich_countries_events)]

metrics_adm0 = evaluate_predictions(top_events_adm0)
print(metrics_adm0)

metrics_adm1 = evaluate_predictions(top_events_adm1.dropna())
print(metrics_adm1)

print(len(top_events_adm0.GID_0.unique()))
print(len(rich_countries_events))

{'mae': 17.5, 'std_error': 28.9, 'quadratic_weighted_kappa': 0.223, 'accuracy': 0.373, 'precision': 0.325, 'recall': 0.593, 'f1': 0.42, 'NPV': 0.847, 'specificity': 0.647}
{'mae': 3.59, 'std_error': 15.6, 'quadratic_weighted_kappa': 0.118, 'accuracy': 0.797, 'precision': 0.106, 'recall': 0.177, 'f1': 0.132, 'NPV': 0.978, 'specificity': 0.961}
40
408


In [19]:
# POOR subset --Naive Wind model
top_events_adm0 = naive_model_adm0[naive_model_adm0["DisNo."].isin(poor_countries_events)]
top_events_adm1 = naive_model_adm1[naive_model_adm1["DisNo."].isin(poor_countries_events)]

metrics_adm0 = evaluate_predictions(top_events_adm0)
print(metrics_adm0)

metrics_adm1 = evaluate_predictions(top_events_adm1.dropna())
print(metrics_adm1)

print(len(top_events_adm0.GID_0.unique()))
print(len(poor_countries_events))

{'mae': 11.6, 'std_error': 27.9, 'quadratic_weighted_kappa': 0.107, 'accuracy': 0.706, 'precision': 0.046, 'recall': 0.5, 'f1': 0.0842, 'NPV': 0.987, 'specificity': 0.78}
{'mae': 2.02, 'std_error': 11.2, 'quadratic_weighted_kappa': 0.337, 'accuracy': 0.912, 'precision': 0.341, 'recall': 0.272, 'f1': 0.302, 'NPV': 0.973, 'specificity': 0.981}
28
385


In [20]:
# POOR subset --Naive historical model
top_events_adm0 = naive_model_adm0_historical[naive_model_adm0_historical["DisNo."].isin(poor_countries_events)]
top_events_adm1 = naive_model_adm1_historical[naive_model_adm1_historical["DisNo."].isin(poor_countries_events)]

metrics_adm0 = evaluate_predictions(top_events_adm0)
print(metrics_adm0)

metrics_adm1 = evaluate_predictions(top_events_adm1.dropna())
print(metrics_adm1)

print(len(top_events_adm0.GID_0.unique()))
print(len(poor_countries_events))

{'mae': 34.0, 'std_error': 35.8, 'quadratic_weighted_kappa': 0.137, 'accuracy': 0.288, 'precision': 0.404, 'recall': 0.801, 'f1': 0.537, 'NPV': 0.733, 'specificity': 0.316}
{'mae': 4.6, 'std_error': 7.64, 'quadratic_weighted_kappa': 0.000816, 'accuracy': 0.192, 'precision': 0.042, 'recall': 0.211, 'f1': 0.07, 'NPV': 0.966, 'specificity': 0.823}
28
385
